In [1]:
from utils import prepare_dataframe

df = prepare_dataframe("merged.csv")
resulting_label_count = df['labels'].nunique()
df


labels
bug            17934
enhancement    16113
clean          15336
performance     5332
refactor        3320
Name: count, dtype: int64
labels
enhancement    15300
bug            15300
clean          15300
Name: count, dtype: int64


,full_repo_name,labels,patch,index
16780,apache/skywalking,enhancement,"@@ -142,4 +142,13 @@ public class StorageModul...",16780
17669,apache/skywalking,enhancement,"@@ -21,6 +21,7 @@\n import java.util.Map;\n im...",17669
39739,apache/netbeans,enhancement,"@@ -19,9 +19,8 @@\n package org.netbeans.modul...",39739
5509,apache/myfaces-tobago,enhancement,"@@ -26,6 +26,7 @@\n \n import jakarta.faces.co...",5509
3099,apache/cloudstack,enhancement,"@@ -42,14 +42,18 @@ public final class Libvirt...",3099
...,...,...,...,...
28870,apache/bookkeeper,clean,"@@ -245,6 +245,7 @@ void unsetSuccessAndSendWr...",28870
30713,apache/netbeans,clean,"@@ -36,7 +36,6 @@\n import org.eclipse.persist...",30713
24323,apache/pulsar,clean,"@@ -18,30 +18,29 @@\n */\n package org.apache...",24323
32419,apache/netbeans,clean,"@@ -77,7 +77,7 @@ public void initialize(Templ...",32419


In [2]:
embedding_directory = 'embeddings/GCB_LHS_embeddings_patch_before_largewithcodedata_15300limit3label'

In [3]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression # Classification
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
import pandas as pd
import torch

# One-hot encoding ile labels sütununu dönüştürme
df_encoded_labels = pd.get_dummies(df, columns=['labels'])
df_encoded_repo = pd.get_dummies(df, columns=['full_repo_name'])

# Özellikler ve etiketler
X_repo = df_encoded_repo.drop(columns=['labels','patch','index'])
X_repo = X_repo.astype(int)

# y = df_encoded.drop(columns=['content_before', 'content_after', 'tokens'])
y = df_encoded_labels.drop(columns=['patch', 'full_repo_name','index'])
y = y.astype(int) # convert it 0,1     instead of true, false 

df['label_tensor'] =[torch.tensor(row,dtype=torch.float32) for row in y.values] 
df['repo_name_tensor'] =[torch.tensor(row,dtype=torch.float32) for row in X_repo.values] 

df = df.drop(columns=['full_repo_name', 'labels','patch'])


print(df['label_tensor'][0] )
print(df['repo_name_tensor'][0] )
df

tensor([0., 0., 1.])
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])


,index,label_tensor,repo_name_tensor
16780,16780,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
17669,17669,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
39739,39739,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
5509,5509,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
3099,3099,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
...,...,...,...
28870,28870,"[tensor(0.), tensor(1.), tensor(0.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
30713,30713,"[tensor(0.), tensor(1.), tensor(0.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
24323,24323,"[tensor(0.), tensor(1.), tensor(0.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
32419,32419,"[tensor(0.), tensor(1.), tensor(0.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."


In [ ]:
from utils import load_embeddings_from_directory_for_df

df = load_embeddings_from_directory_for_df(directory_path=embedding_directory,df=df)

df

In [4]:
X_index = df[['index']]  
# Split the code data into test, train and validation sets
X_index_train, X_index_test = train_test_split(X_index, test_size=0.3, random_state=42) #random_state: 0,3074,42
X_index_test, X_index_val = train_test_split(X_index_test, test_size=0.5, random_state=42)


In [5]:
df_train = df[df['index'].isin(X_index_train['index'].values)].reset_index(drop=True)
df_val = df[df['index'].isin(X_index_val['index'].values)].reset_index(drop=True)
df_test = df[df['index'].isin(X_index_test['index'].values)].reset_index(drop=True)


In [6]:
from torch.utils.data import DataLoader, Dataset
from utils import load_embedding_from_directory

# Custom Dataset Class to handle the dataframe
class CodeDataset(Dataset):
    def __init__(self, df):
        self.df = df
        self.id_list = df['index'].values 

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
    
        index = self.id_list[idx]  
        tensor_3d = load_embedding_from_directory(directory_path= embedding_directory ,id_number=index)
        repo_name_tensor = df.loc[df['index'] == index, 'repo_name_tensor'].values[0] 
        label_tensor = df.loc[df['index'] == index, 'label_tensor'].values[0] 
        
        #Flattening
        tensor_2d = tensor_3d.view(-1,tensor_3d.size(2))
        
        embedding = tensor_2d.view(-1)
        #print(repo_name_tensor.shape)
        #print(embedding.shape)
        return embedding, label_tensor, repo_name_tensor, index


In [7]:
# Create DataLoader

# Creating a DataLoader
def create_dataloader(dataframe, batch_size=32, shuffle=False):
    dataset = CodeDataset(dataframe)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

batch_size = 128
test_loader = create_dataloader(df_test, batch_size=batch_size)
train_loader = create_dataloader(df_train, batch_size=batch_size)
val_loader = create_dataloader(df_val,batch_size=batch_size)

In [8]:
len(train_loader)
df_train


,index,label_tensor,repo_name_tensor
0,39739,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
1,5509,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
2,45680,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
3,827,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
4,53472,"[tensor(0.), tensor(0.), tensor(1.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
...,...,...,...
32125,28870,"[tensor(0.), tensor(1.), tensor(0.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
32126,30713,"[tensor(0.), tensor(1.), tensor(0.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
32127,24323,"[tensor(0.), tensor(1.), tensor(0.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."
32128,32419,"[tensor(0.), tensor(1.), tensor(0.)]","[tensor(0.), tensor(0.), tensor(0.), tensor(0...."


In [9]:

import torch.nn as nn

# Define the ANN model
class CodeDiffNN(nn.Module):
    def __init__(self, code_input_size, repo_input_size, hidden_sizes_pipeline_1,hidden_sizes_pipeline_2, output_size, dropout_rate=0.001):
        super(CodeDiffNN, self).__init__()
        
        # Pipeline for code embeddings
        self.code_fc1 = nn.Linear(code_input_size, hidden_sizes_pipeline_1[0])
        self.code_relu1 = nn.ReLU()
        self.code_dropout1 = nn.Dropout(dropout_rate)
        
        self.code_fc2 = nn.Linear(hidden_sizes_pipeline_1[0], hidden_sizes_pipeline_1[1])
        self.code_relu2 = nn.ReLU()
        self.code_dropout2 = nn.Dropout(dropout_rate)
        
        self.code_fc3 = nn.Linear(hidden_sizes_pipeline_1[1], hidden_sizes_pipeline_1[2])
        self.code_relu3 = nn.ReLU()
        self.code_dropout3 = nn.Dropout(dropout_rate)
        
        
        # Pipeline for repo names
        self.repo_fc1 = nn.Linear(repo_input_size, hidden_sizes_pipeline_2[0])
        self.repo_relu1 = nn.ReLU()
        self.repo_dropout1 = nn.Dropout(dropout_rate)
        
        self.repo_fc2 = nn.Linear(hidden_sizes_pipeline_2[0], hidden_sizes_pipeline_2[1])
        self.repo_relu2 = nn.ReLU()
        self.repo_dropout2 = nn.Dropout(dropout_rate)

        
        # Combining the outputs of both pipelines
        combined_input_size = hidden_sizes_pipeline_1[2] + hidden_sizes_pipeline_2[1]   # Since we're combining two pipelines
        self.fc_combined = nn.Linear(combined_input_size, hidden_sizes_pipeline_2[2])
        self.relu_combined = nn.ReLU()
        self.dropout_combined = nn.Dropout(dropout_rate)

        # Output layer
        self.fc_out = nn.Linear(hidden_sizes_pipeline_2[2], output_size)
        
    def forward(self, code_x, repo_x):
        # Process code embeddings
        code_x = self.code_fc1(code_x)
        code_x = self.code_relu1(code_x)
        code_x = self.code_dropout1(code_x)
        
        code_x = self.code_fc2(code_x)
        code_x = self.code_relu2(code_x)
        code_x = self.code_dropout2(code_x)
        
        code_x = self.code_fc3(code_x)
        code_x = self.code_relu3(code_x)
        code_x = self.code_dropout3(code_x)
        
        # Process repo names
        repo_x = self.repo_fc1(repo_x)
        repo_x = self.repo_relu1(repo_x)
        repo_x = self.repo_dropout1(repo_x)
        
        repo_x = self.repo_fc2(repo_x)
        repo_x = self.repo_relu2(repo_x)
        repo_x = self.repo_dropout2(repo_x)
        
        # Combine both outputs
        combined_x = torch.cat((code_x, repo_x), dim=1)
        
        # Process the combined output
        combined_x = self.fc_combined(combined_x)
        combined_x = self.relu_combined(combined_x)
        combined_x = self.dropout_combined(combined_x)
        
        # Final output
        output = self.fc_out(combined_x)
        return output


In [10]:
'''
import torch
import torch.nn as nn

# Define the ANN model
class CodeDiffNN(nn.Module):
    def __init__(self, code_embedding_dim, repo_input_size, hidden_sizes, output_size, dropout_rate=0.001):
        super(CodeDiffNN, self).__init__()
        
        # Pipeline for code embeddings (Conv2D for 2D input)
        # Set in_channels to the embedding dimension (e.g., 768)
        # Increase out_channels to better handle high-dimensional inputs
        self.code_conv1 = nn.Conv2d(in_channels=code_embedding_dim, out_channels=128, kernel_size=3, padding=1)
        self.code_relu1 = nn.ReLU()
        self.code_dropout1 = nn.Dropout(dropout_rate)
        
        self.code_conv2 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1)
        self.code_relu2 = nn.ReLU()
        self.code_dropout2 = nn.Dropout(dropout_rate)

        # Assuming a fixed input size for flattening (adjust this based on the input size and conv layer operations)
        self.code_flatten = nn.Flatten()

        # Flattened output from conv layers should map to hidden_sizes[1]
        self.code_fc = nn.Linear(256*512, hidden_sizes[1])  # Modify based on input dimensions
        
        # Pipeline for repo names (still using Linear layers for 1D input)
        self.repo_fc1 = nn.Linear(repo_input_size, hidden_sizes[0])
        self.repo_relu1 = nn.ReLU()
        self.repo_dropout1 = nn.Dropout(dropout_rate)
        
        self.repo_fc2 = nn.Linear(hidden_sizes[0], hidden_sizes[1])
        self.repo_relu2 = nn.ReLU()
        self.repo_dropout2 = nn.Dropout(dropout_rate)
        
        # Combining the outputs of both pipelines
        combined_input_size = hidden_sizes[1] * 2  # Since we're combining two pipelines
        self.fc_combined = nn.Linear(combined_input_size, hidden_sizes[2])
        self.relu_combined = nn.ReLU()
        self.dropout_combined = nn.Dropout(dropout_rate)

        # Output layer
        self.fc_out = nn.Linear(hidden_sizes[2], output_size)
        
    def forward(self, code_x, repo_x):
        # Reformat input
        # [batch_size, sequence_length, embedding_dim] -> [batch_size, embedding_dim, sequence_length, 1]
        code_x = code_x.permute(0, 2, 1).unsqueeze(-1)
        
        # Process code embeddings (2D input)
        code_x = self.code_conv1(code_x)
        code_x = self.code_relu1(code_x)
        code_x = self.code_dropout1(code_x)
        
        code_x = self.code_conv2(code_x)
        code_x = self.code_relu2(code_x)
        code_x = self.code_dropout2(code_x)
        
        # Flatten the output from the conv layers before the fully connected layer
        code_x = self.code_flatten(code_x)
        code_x = self.code_fc(code_x)
        
        # Process repo names (1D input)
        repo_x = self.repo_fc1(repo_x)
        repo_x = self.repo_relu1(repo_x)
        repo_x = self.repo_dropout1(repo_x)
        
        repo_x = self.repo_fc2(repo_x)
        repo_x = self.repo_relu2(repo_x)
        repo_x = self.repo_dropout2(repo_x)
        
        # Combine both outputs
        combined_x = torch.cat((code_x, repo_x), dim=1)
        
        # Process the combined output
        combined_x = self.fc_combined(combined_x)
        combined_x = self.relu_combined(combined_x)
        combined_x = self.dropout_combined(combined_x)
        
        # Final output
        output = self.fc_out(combined_x)
        return output
'''

"\nimport torch\nimport torch.nn as nn\n\n# Define the ANN model\nclass CodeDiffNN(nn.Module):\n    def __init__(self, code_embedding_dim, repo_input_size, hidden_sizes, output_size, dropout_rate=0.001):\n        super(CodeDiffNN, self).__init__()\n        \n        # Pipeline for code embeddings (Conv2D for 2D input)\n        # Set in_channels to the embedding dimension (e.g., 768)\n        # Increase out_channels to better handle high-dimensional inputs\n        self.code_conv1 = nn.Conv2d(in_channels=code_embedding_dim, out_channels=128, kernel_size=3, padding=1)\n        self.code_relu1 = nn.ReLU()\n        self.code_dropout1 = nn.Dropout(dropout_rate)\n        \n        self.code_conv2 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1)\n        self.code_relu2 = nn.ReLU()\n        self.code_dropout2 = nn.Dropout(dropout_rate)\n\n        # Assuming a fixed input size for flattening (adjust this based on the input size and conv layer operations)\n        self.c

In [11]:
len(df['repo_name_tensor'][0])

100

In [12]:
import torch.optim as optim
# Define model parameters

torch_size = load_embedding_from_directory(embedding_directory,df['index'][0])[0].shape
code_embedding_dim = torch_size[0]*torch_size[1]

repo_input_size = len(df['repo_name_tensor'][0])
hidden_sizes_pipeline_1 = [1024, 512, 256]
hidden_sizes_pipeline_2 = [512, 256, 256]
output_size = resulting_label_count

device = "cuda:0"
model = CodeDiffNN(code_embedding_dim, repo_input_size, hidden_sizes_pipeline_1,hidden_sizes_pipeline_2, output_size).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-5)

In [13]:
print(resulting_label_count)
print(df['index'][0])
print(load_embedding_from_directory(embedding_directory,df['index'][0])[0].shape)
print(code_embedding_dim)

3
0
torch.Size([512, 768])
393216


In [14]:
import tqdm

# Function to calculate accuracy
def calculate_accuracy(outputs, labels):
    _, predicted = torch.max(outputs, 1)  # Get the index of the max log-probability
    labels_1d = torch.argmax(labels, dim=1) # turn into 1D tensor
    correct = (predicted == labels_1d).sum().item()
    accuracy = correct / labels.size(0)  # Accuracy = number of correct predictions / total number of samples
    return accuracy

def train_with_data_loader(model,loader):
    
    running_loss = 0.0
    running_accuracy = 0.0
    
    for batch_X, batch_y, batch_X_repo, idx in tqdm.notebook.tqdm(loader):
        
        #print(f"batch {batch_X.shape} ")
        #print(f"repo {batch_X_repo.shape} ")
        
        # Forward pass
        batch_y = batch_y.to(device)
        batch_X = batch_X.to(device)
        batch_X_repo = batch_X_repo.to(device)
        
        outputs = model(batch_X, batch_X_repo)
        loss = criterion(outputs, batch_y)
    
        # Backward pass and optimization
        loss.backward()
        optimizer.step()
        
        # Calculate accuracy
        accuracy = calculate_accuracy(outputs, batch_y)
        
        # Update running loss and accuracy
        running_loss += loss.item()
        running_accuracy += accuracy
        
        
    val_loss = running_loss / len(loader)
    val_accuracy = running_accuracy / len(loader)
    
    return val_loss, val_accuracy

In [15]:
def evaluate_with_data_loader(model,loader):
    
    running_loss = 0.0
    running_accuracy = 0.0
    
    for batch_X, batch_y, batch_X_repo, idx in tqdm.notebook.tqdm(loader):
        
        #print(f"batch {batch_X.shape} ")
        #print(f"repo {batch_X_repo.shape} ")
        
        # Forward pass
        batch_y = batch_y.to(device)
        batch_X = batch_X.to(device)
        batch_X_repo = batch_X_repo.to(device)
        
        outputs = model(batch_X, batch_X_repo)
        loss = criterion(outputs, batch_y)
        
        # Calculate accuracy
        accuracy = calculate_accuracy(outputs, batch_y)
        
        # Update running loss and accuracy
        running_loss += loss.item()
        running_accuracy += accuracy
        
        
    val_loss = running_loss / len(loader)
    val_accuracy = running_accuracy / len(loader)
    
    return val_loss, val_accuracy

In [16]:
import copy
import tqdm

# Train the model

# Initialize variables to track the best model and best accuracy
best_accuracy = 0.0
best_model = None

num_epochs = 10

for epoch in tqdm.notebook.tqdm(range(num_epochs)):
    model.train()  # Set the model to training mode
    
    # Print average loss and accuracy for the epoch
    avg_loss,avg_accuracy = train_with_data_loader(model= model,loader= train_loader)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {avg_accuracy:.4f}')

    # Evaluate on validation data
    model.eval()  # Set the model to evaluation mode
    
    with torch.no_grad():  # Disable gradient computation for validation
        val_loss,val_accuracy = evaluate_with_data_loader(model=model,loader=val_loader)
        print(f'Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}')
    
    # Check if the current model has the best validation accuracy
    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        best_model = copy.deepcopy(model.state_dict())  # Save the model with the best accuracy


  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/252 [00:00<?, ?it/s]

Epoch [1/10], Loss: 508.2996, Accuracy: 0.4749


  0%|          | 0/54 [00:00<?, ?it/s]

Validation Loss: 1.1102, Validation Accuracy: 0.3202


...

  0%|          | 0/252 [00:00<?, ?it/s]

Epoch [3/10], Loss: 1.0964, Accuracy: 0.3352


  0%|          | 0/54 [00:00<?, ?it/s]

Validation Loss: 1.1027, Validation Accuracy: 0.3202


  0%|          | 0/252 [00:00<?, ?it/s]

In [ ]:
with torch.no_grad():
    # Load best model from previous training cycle
    model.load_state_dict(best_model)
    print(f'Best model accuracy on validation data: {best_accuracy:.4f}')

    model.eval()
    test_loss,test_accuracy = train_with_data_loader(model=model,loader=test_loader)
    
    print(f'Accuracy on test data: {test_accuracy:.4f}')

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

_, predicted = torch.max(test_outputs.detach().cpu(), 1)  # Get the index of the max log-probability
labels_1d = torch.argmax(y_test.detach().cpu(), dim=1) # turn into 1D tensor

# Convert to numpy arrays
y_test_np = labels_1d.numpy()
predicted_np = predicted.numpy()

# Compute confusion matrix
cm = confusion_matrix(y_test_np, predicted_np)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title('Confusion Matrix for Multi-Class Classification')
plt.show()

In [ ]:
outputs.shape

In [ ]:
### DEPRECATED ###

In [ ]:

# # Örnek tahminler
# sample_data = pd.DataFrame({
#     'file_name': ['ProtoSubject.java','BasicAnnotationProcessor.java'], #1:enhancement, 2:enhancement
#     'full_repo_name': ['google/truth','google/auto']
# })

# # Kodlama işlemleri
# sample_data['file_name_encoded'] = le_class.transform(sample_data['file_name'])
# sample_data['full_repo_name_encoded'] = le_repo.transform(sample_data['full_repo_name'])

# # Özellikleri hazırlama
# X_sample = sample_data[['file_name_encoded', 'full_repo_name_encoded']]
# X_sample = torch.tensor(X_sample.to_numpy(), dtype=torch.float32)

# with torch.no_grad():
#     model.eval()
#     test_outputs = model(X_sample)
#     predicted = torch.sigmoid(test_outputs)  # Sigmoid fonksiyonu uygulanmalı
#     print( predicted)
#     predicted = (predicted > 0.5).float()  # 0.5'ten büyük olanlar 1, diğerleri 0
#     print("after ceiling")
#     print( predicted)